In [1]:
from typing import List, TypedDict, Annotated
import re
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# FIX 1: HuggingFaceEndpoint & ChatHuggingFace import added

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()


d:\Coding\Genarative-AI\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
import os
from langchain_core.output_parsers import StrOutputParser

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=900,chunk_overlap=100)

In [4]:
docs = PyPDFLoader("./documents/book1.pdf").load()

In [5]:
split_docs = splitter.split_documents(docs)

In [6]:
embed_model = HuggingFaceEmbeddings(model= "sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1146.86it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
model = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)


In [8]:
vector_store = FAISS.from_documents(split_docs,embed_model)

In [9]:
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={'k':4})

In [10]:
# retriever sentence striper funtion
def chunks_splitter(text: str) -> list[str]:
    text = re.sub(r"\s+", " ", text).strip()

    # Step 1: sentence boundary দিয়ে আগে ভাগ করো
    raw_sentences = re.split(r"(?<=[.!?])\s+", text)

    # Step 2: ছোট sentence গুলোকে জোড়া লাগিয়ে 200 char-এর chunk বানাও
    chunks = []
    current_chunk = ""

    for s in raw_sentences:
        s = s.strip()
        if not s:
            continue
        # যদি current_chunk-এ s যোগ করলে 200 পার হয়
        if len(current_chunk) + len(s) + 1 > 500 and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = s  # নতুন chunk শুরু হবে এই sentence দিয়ে
        else:
            current_chunk = (current_chunk + " " + s).strip()

    # শেষে যা বাকি থাকবে সেটাও রাখো
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks


In [11]:
upper_th = 0.7
lower_th = 0.3

In [12]:
class State(TypedDict):
    question:str
    docs:list[Document]
    ans:str

    all_strips:list[str]
    kept_strips:list[str]
    refined_context:str
    good_docs:list[Document]
    verdict:str
    reason:str

    web_docs: list[Document]
    rewrite_query:str

    answer:str


In [13]:
def retrieve_node(state):
    q = state['question']
    return {'docs':retriever.invoke(q)}

In [14]:
class DocEvalScore(BaseModel):
    score: float
    reason: str

doc_eval_prompt = ChatPromptTemplate.from_messages(
    [
        ('system',"You are a strict retrieval evaluator for RAG.\n"
            "You will be given ONE retrieved chunk and a question.\n"
            "Return a relevance score in [0.0, 1.0].\n"
            "- 1.0: chunk alone is sufficient to answer fully/mostly\n"
            "- 0.0: chunk is irrelevant\n"
            "Be conservative with high scores.\n"
            "Also return a short reason.\n"
            "You MUST output valid JSON only."
            "1. \"score\" (a float between 0.0 and 1.0)\n"
            "2. \"reason\" (a short string explaining the score)"
            ),
        ('human','Question :{question}\n\nchunk:\n{chunk}')
    ]
)

doc_eval_chain = doc_eval_prompt | model.with_structured_output(DocEvalScore,method='json_mode')

def doc_eval_node(state:State) -> State:
    q = state['question']

    scores:list[float] = []
    reasons:list[str] = []
    good:list[Document] = []

    for d in state['docs']:
        out = doc_eval_chain.invoke({'question':q,'chunk':d.page_content})
        scores.append(out.score)
        reasons.append(out.reason)

        if out.score > lower_th:
            good.append(d)

    if any(s > upper_th for s in scores):
        return{
            'good_docs': good,
            'verdict': 'CORRECT',
            'reason':f'At least one retrieved chunk scored > {upper_th}.'
        }
    
    if len(scores) > 0 and all(s < lower_th for s in scores):
        why = 'No chunk was sufficient'
        return {
            'good_docs': [],
            'verdict':"INCORRECT",
            "reason": f"All retrieved chunks scored < {lower_th}. {why}"
        }

    why = 'Mixed relevance signals'
    return {
        'good_docs': good,
        'verdict': "AMBIGUOUS",
        'reason': f"No chunk scored > {upper_th}, but not all were < {lower_th}. {why}",
    }
    
    

In [15]:
filter_prompt = ChatPromptTemplate(
    [
          ("system", 
         "You are a strict relevance filter. \n"
         "Does the sentence directly help answer the question? \n"
         "Reply with ONLY one word: YES or NO. Nothing else."
        ),
        ('human','question:{question}\n\nsentence:{sentence}')
    ]
)

filter_chain = filter_prompt | model | StrOutputParser() | (lambda x:"YES" in x.strip().upper())

In [26]:
def refine_node(state:State) -> State:
    q = state['question']
    good_docs = state.get('good_docs',[])
    web_docs = state.get('web_docs',[])
    all_split_chunks = []
    all_strips = []
    all_output = []
    kept_strips = []
    all_input_for_filter = []

    if state.get('verdict') == "CORRECT":
        final_docs = good_docs
    elif state.get('verdict') == "INCORRECT":
        final_docs = web_docs
    else:
        final_docs = state['good_docs'] + state['web_docs']

    for doc in final_docs:
        strips = chunks_splitter(doc.page_content)
        all_split_chunks.extend(strips)

        for s in strips:
            all_input_for_filter.append({'question':q,'sentence':s})
            all_strips.append(s)

    all_output = filter_chain.batch(all_input_for_filter)

    for s,r in zip(all_strips,all_output):
        if r:
            kept_strips.append(s)
    
    refined_context = "\n\n".join(kept_strips)

    return {
        'kept_strips':kept_strips,
        'refined_context':refined_context,
        "all_strips":all_strips

    }

In [53]:

load_dotenv()
# WEB Search Node
from langchain_community.tools import TavilySearchResults

class WebQuery(BaseModel):
    query:str

rewrite_prompt = ChatPromptTemplate(
    [
        ("system",
    "Rewrite the user question into a web search query composed of keywords. \n"
    "Rules: \n"
    "- Keep it short (6-14 words). \n"
    "- Focus on the core topic, strip filler words. \\n"  
    "- If the question implies recency (e.g., recent/latest/last week/last month), add a constraint like (last 30 days).\n"
    "- Do NOT answer the question. \n"
    "- Return JSON with a single key: query",
    ),
    ("human", "Question: {question}")
    ]
)

rewrite_chain = rewrite_prompt | model.with_structured_output(WebQuery,method='json_mode')


#Re Write Node
def rewrite_query_node(state:State) -> State:
    out = rewrite_chain.invoke({'question':state['question']})
    return{
        'rewrite_query':out.query
        
    }


tavily = TavilySearchResults(max_results = 2)

def web_search_node(state:State) -> State:
    q = state.get('rewrite_query') or state['rewrite_query']
    result = tavily.invoke({'query':q})

    web_docs = []
    for r in result or []:
        title = r.get('title','')
        url = r.get('url','')
        content = r.get('content','') or r.get('snippet','')

        text = f'TITLE: {title}\nURL: {url}\nCONTENT: {content}'

        web_docs.append(Document(page_content=text, metadata={'url':url, 'title':title}))
    return{
        'web_docs':web_docs
    }

In [54]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ('system',"Answer only from the context. If not in contex, say you don't know",),
        ('human', "Question : {question}\n\nContext:\n{context}")
    ]
)

generate_chain = answer_prompt | model | StrOutputParser()

def generate(state):
    question = state['question']
    context = state['refined_context']
    answer = generate_chain.invoke({'question':question,'context':context})
    return {'answer':answer}


In [55]:




def route_after_eval(state:State) -> str:
    if state['verdict'] == "CORRECT":
        return 'refine'
    else:
        return "rewrite_query"



In [56]:

g = StateGraph(State)
g.add_node("retrieve", retrieve_node)
g.add_node('eval_each_doc',doc_eval_node)

g.add_node('rewrite_query',rewrite_query_node)
g.add_node('web_search',web_search_node)

g.add_node("refine", refine_node)
g.add_node("generate", generate) 


g.add_edge(START, "retrieve")
g.add_edge("retrieve", "eval_each_doc")

g.add_conditional_edges(
    "eval_each_doc",
    route_after_eval,
    {"refine": "refine", "rewrite_query": "rewrite_query"}
)


g.add_edge('rewrite_query','web_search')
g.add_edge('web_search','refine')

g.add_edge("refine", "generate")
g.add_edge("generate", END)


app = g.compile()
print("Graph compiled successfully!")


Graph compiled successfully!


In [57]:
result = app.invoke({
    "question": "What is neural network?"

})
print("VERDICT: ",result['verdict'])
print("REASON: ",result['reason'])
print("Answer:", result["answer"])



VERDICT:  CORRECT
REASON:  At least one retrieved chunk scored > 0.7.
Answer: A neural network is a multilayer perceptron (MLP) that uses continuous sigmoidal nonlinearities in the hidden units, making it differentiable with respect to the network parameters.


In [52]:
print(result['rewrite_query'])
print(result['web_docs'])


Neural Network Definition
[Document(metadata={'url': 'https://aws.amazon.com/what-is/neural-network/', 'title': 'What is a Neural Network? - AWS'}, page_content='TITLE: What is a Neural Network? - AWS\nURL: https://aws.amazon.com/what-is/neural-network/\nCONTENT: ## What is a neural network?\n\nA neural network is a method in artificial intelligence (AI) that teaches computers to process data in a way that is inspired by the human brain. It is a type of machine learning (ML) process, called deep learning, that uses interconnected nodes or neurons in a layered structure that resembles the human brain. It creates an adaptive system that computers use to learn from their mistakes and improve continuously. Thus, artificial neural networks attempt to solve complicated problems, like summarizing documents or recognizing faces, with greater accuracy.\n\n## Why are neural networks important?'), Document(metadata={'url': 'https://www.geeksforgeeks.org/deep-learning/neural-networks-a-beginners-g